In [ ]:
!pip install -q torch torchvision finrl yfinance stable-baselines3 gymnasium

import torch
print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
# Clone official DSAC repo and install deps
!git clone https://github.com/xtma/dsac.git
!pip install finrl yfinance

import sys
sys.path.insert(0, '/kaggle/working/dsac')
sys.path.insert(0, '/kaggle/working/dsac/rlkit')

In [ ]:
!git clone https://github.com/AI4Finance-Foundation/FinRL.git
!pip install -r /kaggle/working/FinRL/requirements.txt

In [ ]:
!pip install stable_baselines3 alpaca_trade_api exchange_calendars stockstats wrds -q

In [ ]:
!pip install websockets --upgrade


In [ ]:
import finrl.meta.preprocessor.yahoodownloader as yd_module
import yfinance as yf
import pandas as pd

def patched_fetch_data(self, proxy=None):
    dfs = []
    for tic in self.ticker_list:
        try:
            df = yf.download(tic, start=self.start_date, end=self.end_date, auto_adjust=True, progress=False)
            if df.empty:
                print(f"{tic}: empty"); continue
            # Flatten MultiIndex columns if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [col[0].lower() for col in df.columns]
            else:
                df.columns = [c.lower() for c in df.columns]
            df['tic'] = tic
            df = df.reset_index().rename(columns={'date': 'date', 'index': 'date', 'Date': 'date'})
            df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None).dt.strftime('%Y-%m-%d')
            df = df[['date', 'open', 'high', 'low', 'close', 'volume', 'tic']].dropna()
            dfs.append(df)
            print(f"{tic}: {len(df)} rows")
        except Exception as e:
            print(f"{tic} failed: {e}")
    if not dfs:
        raise ValueError("no data is fetched.")
    return pd.concat(dfs).sort_values(['date', 'tic']).reset_index(drop=True)

yd_module.YahooDownloader.fetch_data = patched_fetch_data

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
TICKER_LIST = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA']
df_raw = YahooDownloader(start_date='2016-01-01', end_date='2023-12-31', ticker_list=TICKER_LIST).fetch_data()
print(df_raw.shape)
print(df_raw.head())

In [ ]:
# Download and prep FinRL data 
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.config import INDICATORS

TICKER_LIST = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA']
df_raw = YahooDownloader(start_date='2016-01-01', end_date='2023-12-31', ticker_list=TICKER_LIST).fetch_data()


In [ ]:
fe = FeatureEngineer(use_technical_indicator=True, tech_indicator_list=INDICATORS, use_vix=True, use_turbulence=True)
df = fe.preprocess_data(df_raw).fillna(0)
df_train = data_split(df, '2016-01-01', '2021-12-31')
df_test  = data_split(df, '2022-01-01', '2023-12-31')

stock_dim   = len(TICKER_LIST)
state_space = 1 + 2 * stock_dim + len(INDICATORS) * stock_dim
ENV_KWARGS = dict(
    hmax=100, initial_amount=1_000_000,
    num_stock_shares=[0]*stock_dim,
    buy_cost_pct=[0.001]*stock_dim, sell_cost_pct=[0.001]*stock_dim,
    reward_scaling=1e-2, state_space=state_space, stock_dim=stock_dim,
    tech_indicator_list=INDICATORS, action_space=stock_dim,
    make_plots=False, print_verbosity=10,
)
train_env = StockTradingEnv(df=df_train, **ENV_KWARGS)
test_env  = StockTradingEnv(df=df_test,  **ENV_KWARGS)

In [ ]:
#Patch rlkit's get_dim to handle gymnasium 
import gymnasium
import gym
from rlkit.envs import env_utils

def patched_get_dim(space):
    if isinstance(space, (gym.spaces.Discrete, gymnasium.spaces.Discrete)):
        return space.n
    elif isinstance(space, (gym.spaces.Box, gymnasium.spaces.Box)):
        return int(np.prod(space.shape))
    elif hasattr(space, 'flat_dim'):
        return space.flat_dim
    else:
        raise TypeError("Unknown space: {}".format(space))

env_utils.get_dim = patched_get_dim

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Fit scaler on training data observations
# Collect sample obs from env
sample_obs = []
obs, _ = train_env.reset()
for _ in range(2000):
    sample_obs.append(obs)
    action = train_env.action_space.sample()
    obs, _, term, trunc, _ = train_env.step(action)
    if term or trunc:
        obs, _ = train_env.reset()

scaler = StandardScaler()
scaler.fit(np.array(sample_obs))

# The FinRL wrapper
class FinRLWrapper(gym.Env):
    def __init__(self, env, scaler=None):
        self._env = env
        self._scaler = scaler
        obs_shape = env.observation_space.shape
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=obs_shape, dtype=np.float32
        )
        self.action_space = gym.spaces.Box(
            low=env.action_space.low,
            high=env.action_space.high,
            shape=env.action_space.shape, dtype=np.float32
        )

    def _normalize(self, obs):
        obs = np.array(obs, dtype=np.float32)
        if self._scaler is not None:
            obs = self._scaler.transform(obs.reshape(1, -1)).flatten().astype(np.float32)
        return obs

    def reset(self):
        obs, _ = self._env.reset()
        return self._normalize(obs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self._env.step(action)
        return self._normalize(obs), float(reward), terminated or truncated, info

    def render(self, mode='human'):
        pass

# Wrappers with scaler
wrapped_train = FinRLWrapper(train_env, scaler=scaler)
wrapped_test  = FinRLWrapper(StockTradingEnv(df=df_test, **ENV_KWARGS), scaler=scaler)

In [ ]:

import rlkit.torch.pytorch_util as ptu
from rlkit.torch.sac.policies import TanhGaussianPolicy
import torch
import numpy as np

original_get_actions = TanhGaussianPolicy.get_actions

def patched_get_actions(self, obs_np, **kwargs):
    obs_np = np.array(obs_np, dtype=np.float32)
    return original_get_actions(self, obs_np, **kwargs)

TanhGaussianPolicy.get_actions = patched_get_actions

In [ ]:
!pip install gtimer

In [ ]:
import rlkit.torch.dsac.dsac as dsac_module
import rlkit.torch.pytorch_util as ptu
import torch

original_train_from_torch = dsac_module.DSACTrainer.train_from_torch

def patched_train_from_torch(self, batch):
    # Convert batch to tensors
    tensor_batch = {
        k: torch.FloatTensor(v).to(ptu.device) if not isinstance(v, torch.Tensor)
        else v.to(ptu.device)
        for k, v in batch.items()
    }

    obs = tensor_batch['observations']

    # Compute new_actions with fresh graph for alpha update only
    with torch.no_grad():
        _, _, _, log_pi_alpha, *_ = self.policy(obs, reparameterize=False, return_log_prob=True)

    if self.use_automatic_entropy_tuning:
        alpha_loss = -(self.log_alpha.exp() * (log_pi_alpha + self.target_entropy).detach()).mean()
        self.alpha_optimizer.zero_grad()
        alpha_loss.backward()
        self.alpha_optimizer.step()

    # Temporarily disable auto entropy so original method skips alpha update
    original_auto = self.use_automatic_entropy_tuning
    self.use_automatic_entropy_tuning = False
    original_alpha = self.alpha
    self.alpha = self.log_alpha.exp().item() if original_auto else self.alpha

    result = original_train_from_torch(self, tensor_batch)

    # Restore
    self.use_automatic_entropy_tuning = original_auto
    self.alpha = original_alpha

    return result

dsac_module.DSACTrainer.train_from_torch = patched_train_from_torch

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
import rlkit.core.rl_algorithm as rl_alg_module


save_config = {'policy_dir': '/kaggle/working/dsac_policies'}
KEEP_LAST_N = 20
training_log = defaultdict(list)

if not getattr(rl_alg_module.BaseRLAlgorithm._end_epoch, '_is_patched', False):
    _true_original_end_epoch = rl_alg_module.BaseRLAlgorithm._end_epoch


    def logging_end_epoch(self, epoch):
        def compute_iqn_cvar_epoch(trainer, eval_paths, scaler, alpha=0.25, n_taus=500):
            """
            Compute CVaR analytically from IQN critic using observations
            from eval paths — no need to re-rollout the environment.
            """
            all_cvars = []
            
            for path in eval_paths:
                obs_array     = path['observations']   # (T, obs_dim)
                action_array  = path['actions']        # (T, action_dim)
                
                obs_t    = torch.FloatTensor(obs_array).to(ptu.device)
                action_t = torch.FloatTensor(action_array).to(ptu.device)
                T        = obs_t.shape[0]
                
                with torch.no_grad():
                    # Sample taus only in [0, alpha] — CVaR tail
                    taus = torch.rand(T, n_taus).to(ptu.device) * alpha
                    
                    z1 = trainer.zf1(obs_t, action_t, taus)  # (T, n_taus)
                    z2 = trainer.zf2(obs_t, action_t, taus)  # (T, n_taus)
                    z  = torch.min(z1, z2)                   # (T, n_taus)
                    
                    # CVaR per timestep = mean over tail quantiles
                    cvar_per_step = z.mean(dim=1)            # (T,)
                    all_cvars.append(cvar_per_step.mean().item())
            
            return np.mean(all_cvars)
            
        def discounted_return(rewards, gamma=0.99):
            G = 0
            for r in reversed(rewards):
                G = r + gamma * G
            return G
            
        stats = self.trainer.eval_statistics
        if stats:
            for k, v in stats.items():
                training_log[k].append(float(v) if v is not None else np.nan)
            training_log['epoch'].append(epoch)

            expl = self.expl_data_collector.get_diagnostics()
            for k, v in expl.items():
                try: training_log[f'expl/{k}'].append(float(v))
                except: pass

            expl_paths = self.expl_data_collector.get_epoch_paths()
            if expl_paths:
                expl_returns = [sum(p['rewards']) for p in expl_paths]
                expl_rewards = np.concatenate([p['rewards'] for p in expl_paths])
                training_log['expl/Average Returns'].append(np.mean(expl_returns))
                training_log['expl/Rewards Mean'].append(np.mean(expl_rewards))
                training_log['expl/Rewards Std'].append(np.std(expl_rewards))

            evl = self.eval_data_collector.get_diagnostics()
            for k, v in evl.items():
                try: training_log[f'eval/{k}'].append(float(v))
                except: pass

            eval_paths = self.eval_data_collector.get_epoch_paths()
            if eval_paths:
                eval_returns = [sum(p['rewards']) for p in eval_paths]
                eval_rewards = np.concatenate([p['rewards'] for p in eval_paths])
                training_log['eval/Average Returns'].append(np.mean(eval_returns))
                training_log['eval/Rewards Mean'].append(np.mean(eval_rewards))
                training_log['eval/Rewards Std'].append(np.std(eval_rewards))
                eval_actions = np.concatenate([p['actions'] for p in eval_paths])
                training_log['eval/Actions Mean'].append(np.mean(eval_actions))
                training_log['eval/Actions Std'].append(np.std(eval_actions))
                iqn_cvar = compute_iqn_cvar_epoch(
                    self.trainer, eval_paths, scaler, alpha=0.25
                )
                training_log['eval/IQN_CVaR_25'].append(iqn_cvar)
                ep_returns = np.array([discounted_return(p['rewards']) for p in eval_paths])
                if len(ep_returns) > 1:
                    threshold = np.percentile(ep_returns, 25)
                    emp_cvar  = ep_returns[ep_returns <= threshold].mean()
                else:
                    emp_cvar  = float(ep_returns[0])
                training_log['eval/Empirical_CVaR_25'].append(float(emp_cvar))
            # Save policy checkpoint
            policy_dir = save_config['policy_dir']
            os.makedirs(policy_dir, exist_ok=True)
            ckpt_path = os.path.join(policy_dir, f'policy_epoch_{epoch:04d}.pt')
            torch.save({
                'epoch': epoch,
                'policy':        self.trainer.policy.state_dict(),
                'target_policy': self.trainer.target_policy.state_dict(),
                'zf1':           self.trainer.zf1.state_dict(),
                'zf2':           self.trainer.zf2.state_dict(),
            }, ckpt_path)

            # Delete old checkpoints, keep only last KEEP_LAST_N
            all_ckpts = sorted([
                f for f in os.listdir(policy_dir) if f.startswith('policy_epoch_')
            ])
            for old in all_ckpts[:-KEEP_LAST_N]:
                os.remove(os.path.join(policy_dir, old))
            print(f"Epoch {epoch} | Saved to {policy_dir} | {min(len(all_ckpts), KEEP_LAST_N)} kept")

        _true_original_end_epoch(self, epoch)

    logging_end_epoch._is_patched = True
    rl_alg_module.BaseRLAlgorithm._end_epoch = logging_end_epoch
    print("Logging hook installed with policy saving.")
else:
    print("Hook already installed — skipping.")

In [ ]:
import torch
import numpy as np
import importlib
import gtimer as gt
import rlkit.torch.sac.policies as sac_policies
import rlkit.torch.dsac.dsac as dsac_module
import rlkit.torch.pytorch_util as ptu
from rlkit.launchers.launcher_util import setup_logger
from rlkit.torch.dsac.dsac import DSACTrainer
from rlkit.torch.dsac.networks import QuantileMlp
from rlkit.samplers.data_collector import MdpPathCollector
from rlkit.data_management.env_replay_buffer import EnvReplayBuffer
from rlkit.torch.torch_rl_algorithm import TorchBatchRLAlgorithm

# Patch 
importlib.reload(sac_policies)
importlib.reload(dsac_module)
from rlkit.torch.sac.policies import TanhGaussianPolicy, MakeDeterministic
from rlkit.torch.dsac.dsac import DSACTrainer

# Patch fix numpy obs input
_original_get_actions = TanhGaussianPolicy.get_actions
def patched_get_actions(self, obs_np, **kwargs):
    obs_np = np.array(obs_np, dtype=np.float32)
    return _original_get_actions(self, obs_np, **kwargs)
TanhGaussianPolicy.get_actions = patched_get_actions

# Patch fix numpy batch + unblock policy gradient
_original_train_from_torch = DSACTrainer.train_from_torch
def patched_train_from_torch(self, batch):
    # Convert all batch entries to tensors
    batch = {
        k: torch.FloatTensor(v).to(ptu.device) if not isinstance(v, torch.Tensor)
        else v.to(ptu.device)
        for k, v in batch.items()
    }
    # Resample new_actions with fresh graph after any alpha backward

    return _original_train_from_torch(self, batch)
DSACTrainer.train_from_torch = patched_train_from_torch

ptu.set_gpu_mode(torch.cuda.is_available())

env      = wrapped_train
eval_env = FinRLWrapper(StockTradingEnv(df=df_test, **ENV_KWARGS), scaler=scaler)

obs_dim    = env.observation_space.low.size
action_dim = env.action_space.low.size
M = 256



In [ ]:
import pickle

SEEDS = [321]
seed_results = {}
all_training_logs = {}

for seed in SEEDS:
    print(f"\n{'='*50}\nStarting seed {seed}\n{'='*50}")
    
    # Reset training log for seed 
    training_log.clear()
    save_config['policy_dir'] = f'/kaggle/working/dsac_policies_seeded{seed}'
    # ── Set seeds ────────────────────────────────────
    torch.manual_seed(seed)
    np.random.seed(seed)
    import random; random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    # Build networks 
    zf1        = QuantileMlp(input_size=obs_dim + action_dim, output_size=1, hidden_sizes=[M, M])
    zf2        = QuantileMlp(input_size=obs_dim + action_dim, output_size=1, hidden_sizes=[M, M])
    target_zf1 = QuantileMlp(input_size=obs_dim + action_dim, output_size=1, hidden_sizes=[M, M])
    target_zf2 = QuantileMlp(input_size=obs_dim + action_dim, output_size=1, hidden_sizes=[M, M])
    policy        = TanhGaussianPolicy(obs_dim=obs_dim, action_dim=action_dim, hidden_sizes=[M, M])
    target_policy = TanhGaussianPolicy(obs_dim=obs_dim, action_dim=action_dim, hidden_sizes=[M, M])
    eval_policy   = MakeDeterministic(policy)

    eval_collector = MdpPathCollector(eval_env, eval_policy)
    expl_collector = MdpPathCollector(env, policy)
    replay_buffer  = EnvReplayBuffer(1_000_000, env)

    trainer = DSACTrainer(
        env=env, policy=policy, target_policy=target_policy,
        zf1=zf1, zf2=zf2, target_zf1=target_zf1, target_zf2=target_zf2,
        discount=0.99, reward_scale=1.0,
        policy_lr=3e-4, zf_lr=3e-4,
        tau_type='iqn', num_quantiles=32,
        risk_type='cvar', risk_param=0.25,
        soft_target_tau=0.005,
        use_automatic_entropy_tuning=False,
        alpha=0.2, clip_norm=1.0,
    )

    algorithm = TorchBatchRLAlgorithm(
        trainer=trainer,
        exploration_env=env, evaluation_env=eval_env,
        exploration_data_collector=expl_collector,
        evaluation_data_collector=eval_collector,
        replay_buffer=replay_buffer,
        batch_size=256, max_path_length=500,
        num_epochs=200,
        num_eval_steps_per_epoch=500,
        num_expl_steps_per_train_loop=500,
        num_trains_per_train_loop=500,
        min_num_steps_before_training=5000,
    )

    

    gt.reset_root()
    setup_logger(f'dsac_cvar_seed{seed}', variant={})
    algorithm.to(ptu.device)
    algorithm.train()

    # Save training log 
    all_training_logs[seed] = dict(training_log)
    with open(f'/kaggle/working/dsac_training_log_seed{seed}.pkl', 'wb') as f:
        pickle.dump(dict(training_log), f)
    print(f"Training log saved for seed {seed}")

    # Backtest
    bt_env = StockTradingEnv(df=df_test, **ENV_KWARGS)
    obs, _ = bt_env.reset()
    obs = scaler.transform(np.array(obs).reshape(1,-1)).flatten().astype(np.float32)
    pv, done = [bt_env.initial_amount], False
    while not done:
        action, _ = eval_policy.get_action(obs)
        obs, r, term, trunc, _ = bt_env.step(action)
        obs = scaler.transform(np.array(obs).reshape(1,-1)).flatten().astype(np.float32)
        pv.append(bt_env.asset_memory[-1])
        done = term or trunc

    pv  = np.array(pv)
    ret = np.diff(pv) / pv[:-1]
    seed_results[seed] = {
        'total_return': (pv[-1]/pv[0] - 1) * 100,
        'sharpe':       ret.mean() / (ret.std() + 1e-9) * np.sqrt(252),
        'max_dd':       (pv / np.maximum.accumulate(pv) - 1).min() * 100,
        'cvar_5':       ret[ret <= np.percentile(ret, 5)].mean() * 100,
        'var_5':        np.percentile(ret, 5) * 100,
        'calmar':       ((pv[-1]/pv[0]-1)*100) / (abs((pv/np.maximum.accumulate(pv)-1).min()*100) + 1e-9),
        'win_rate':     (ret > 0).mean() * 100,
    }

    # Save metrics
    with open(f'/kaggle/working/dsac_metrics_seed{seed}.pkl', 'wb') as f:
        pickle.dump(seed_results[seed], f)
    print(f"Seed {seed} | Return: {seed_results[seed]['total_return']:+.2f}% | Sharpe: {seed_results[seed]['sharpe']:.3f}")

# Final summary 
df_results = pd.DataFrame(seed_results).T
print("\n=== DSAC + CVaR Results ===")
print(df_results.to_string())
print("\n=== Mean ± Std ===")
print(df_results.agg(['mean', 'std']).to_string())

# Save full summary
df_results.to_csv('/kaggle/working/dsac_cvar_summary.csv')
with open('/kaggle/working/dsac_all_training_logs.pkl', 'wb') as f:
    pickle.dump(all_training_logs, f)
print("All results saved.")

In [ ]:
# # ── 4. Backtest portfolio analysis ───────────────────────────────
# from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
# import matplotlib.pyplot as plt

# bt_env = StockTradingEnv(df=df_test, **ENV_KWARGS)
# obs, _ = bt_env.reset()
# obs = scaler.transform(np.array(obs).reshape(1, -1)).flatten().astype(np.float32)

# portfolio_values, actions_taken, rewards_taken = [bt_env.initial_amount], [], []
# done = False

# while not done:
#     action, _ = eval_policy.get_action(obs)  # obs is already (state_dim,) numpy
#     obs, r, term, trunc, _ = bt_env.step(action)
#     obs = scaler.transform(np.array(obs).reshape(1, -1)).flatten().astype(np.float32)
#     portfolio_values.append(bt_env.asset_memory[-1])
#     actions_taken.append(action)
#     rewards_taken.append(r)
#     done = term or trunc

# pv      = np.array(portfolio_values)
# actions = np.array(actions_taken)
# returns = np.diff(pv) / (pv[:-1] + 1e-9)

# # Metrics
# total_ret  = (pv[-1] / pv[0] - 1) * 100
# ann_ret    = ((pv[-1] / pv[0]) ** (252 / len(returns)) - 1) * 100
# sharpe     = returns.mean() / (returns.std() + 1e-9) * np.sqrt(252)
# max_dd     = (pv / np.maximum.accumulate(pv) - 1).min() * 100
# var_95     = np.percentile(returns, 5) * 100
# cvar_95    = returns[returns <= np.percentile(returns, 5)].mean() * 100
# calmar     = ann_ret / (abs(max_dd) + 1e-9)
# win_rate   = (returns > 0).mean() * 100

# print("=" * 45)
# print("        BACKTEST RESULTS (Test Set)")
# print("=" * 45)
# print(f"  Total Return      : {total_ret:+.2f}%")
# print(f"  Annualised Return : {ann_ret:+.2f}%")
# print(f"  Sharpe Ratio      : {sharpe:.3f}")
# print(f"  Calmar Ratio      : {calmar:.3f}")
# print(f"  Max Drawdown      : {max_dd:.2f}%")
# print(f"  VaR  (5%)         : {var_95:.4f}%")
# print(f"  CVaR (5%)         : {cvar_95:.4f}%")
# print(f"  Win Rate          : {win_rate:.1f}%")
# print("=" * 45)

# fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# # Portfolio curve
# axes[0].plot(pv / pv[0], color='teal', lw=1.5)
# axes[0].axhline(1.0, color='grey', linestyle='--', lw=0.8)
# dd = pv / np.maximum.accumulate(pv) - 1
# axes[0].fill_between(range(len(pv)), pv/pv[0], 1.0,
#                      where=(pv/pv[0] < 1.0), alpha=0.2, color='red', label='Drawdown')
# axes[0].set_title(f'Portfolio Value — Total Return: {total_ret:+.2f}%  |  Sharpe: {sharpe:.3f}  |  MaxDD: {max_dd:.2f}%')
# axes[0].set_ylabel('Normalised Value'); axes[0].legend(); axes[0].grid(alpha=0.3)

# # Drawdown curve
# axes[1].fill_between(range(len(dd)), dd * 100, 0, alpha=0.6, color='red')
# axes[1].set_title('Drawdown (%)')
# axes[1].set_ylabel('Drawdown %'); axes[1].grid(alpha=0.3)

# # Action heatmap (per stock over time)
# im = axes[2].imshow(actions.T, aspect='auto', cmap='RdYlGn',
#                     vmin=-1, vmax=1, interpolation='nearest')
# axes[2].set_title('Actions Heatmap (green=buy, red=sell)')
# axes[2].set_xlabel('Timestep'); axes[2].set_ylabel('Stock')
# axes[2].set_yticks(range(len(TICKER_LIST))); axes[2].set_yticklabels(TICKER_LIST)
# plt.colorbar(im, ax=axes[2])

# plt.tight_layout()
# plt.savefig('/content/drive/MyDrive/dsac_policies_run3/dsac_cvar_backtest.png', dpi=150, bbox_inches='tight')
# plt.show()

# # Return distribution
# fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
# axes2[0].hist(returns * 100, bins=50, color='teal', edgecolor='white', alpha=0.8)
# axes2[0].axvline(var_95,  color='orange', linestyle='--', lw=1.5, label=f'VaR 5%: {var_95:.3f}%')
# axes2[0].axvline(cvar_95, color='red',    linestyle='--', lw=1.5, label=f'CVaR 5%: {cvar_95:.3f}%')
# axes2[0].set_title('Return Distribution'); axes2[0].set_xlabel('Daily Return (%)')
# axes2[0].legend(); axes2[0].grid(alpha=0.3)

# # Cumulative return
# axes2[1].plot(np.cumprod(1 + returns) - 1, color='teal')
# axes2[1].set_title('Cumulative Return'); axes2[1].set_xlabel('Timestep')
# axes2[1].set_ylabel('Cumulative Return'); axes2[1].grid(alpha=0.3)

# plt.tight_layout()
# plt.savefig('/content/drive/MyDrive/dsac_policies_run3/dsac_cvar_returns.png', dpi=150, bbox_inches='tight')
# plt.show()

In [ ]:
# import torch
# import numpy as np
# import importlib
# import gtimer as gt
# import rlkit.torch.sac.policies as sac_policies
# import rlkit.torch.pytorch_util as ptu
# from rlkit.envs.wrappers import NormalizedBoxEnv
# from rlkit.launchers.launcher_util import setup_logger
# from rlkit.torch.dsac.dsac import DSACTrainer
# from rlkit.torch.dsac.networks import QuantileMlp
# from rlkit.samplers.data_collector import MdpPathCollector
# from rlkit.data_management.env_replay_buffer import EnvReplayBuffer
# from rlkit.torch.torch_rl_algorithm import TorchBatchRLAlgorithm

# # ── Step 1: reload and patch BEFORE creating any policy objects ──
# importlib.reload(sac_policies)
# from rlkit.torch.sac.policies import TanhGaussianPolicy, MakeDeterministic

# _original_get_actions = TanhGaussianPolicy.get_actions

# def patched_get_actions(self, obs_np, **kwargs):
#     obs_np = np.array(obs_np, dtype=np.float32)
#     return _original_get_actions(self, obs_np, **kwargs)

# TanhGaussianPolicy.get_actions = patched_get_actions

# # ── Step 2: build envs ───────────────────────────────────────────
# ptu.set_gpu_mode(torch.cuda.is_available())

# env      = wrapped_train
# eval_env = FinRLWrapper(StockTradingEnv(df=df_test, **ENV_KWARGS), scaler=scaler)

# obs_dim    = env.observation_space.low.size
# action_dim = env.action_space.low.size
# M = 256

# # ── Step 3: build networks ───────────────────────────────────────
# zf1        = QuantileMlp(input_size=obs_dim + action_dim, output_size=1, hidden_sizes=[M, M])
# zf2        = QuantileMlp(input_size=obs_dim + action_dim, output_size=1, hidden_sizes=[M, M])
# target_zf1 = QuantileMlp(input_size=obs_dim + action_dim, output_size=1, hidden_sizes=[M, M])
# target_zf2 = QuantileMlp(input_size=obs_dim + action_dim, output_size=1, hidden_sizes=[M, M])

# # target_policy must be a full TanhGaussianPolicy, not MakeDeterministic
# policy        = TanhGaussianPolicy(obs_dim=obs_dim, action_dim=action_dim, hidden_sizes=[M, M])
# target_policy = TanhGaussianPolicy(obs_dim=obs_dim, action_dim=action_dim, hidden_sizes=[M, M])  # separate network
# eval_policy   = MakeDeterministic(policy)

# # ── Step 4: build algorithm ──────────────────────────────────────
# eval_collector = MdpPathCollector(eval_env, eval_policy)
# expl_collector = MdpPathCollector(env, policy)
# replay_buffer  = EnvReplayBuffer(1_000_000, env)

# trainer = DSACTrainer(
#     env=env, policy=policy, target_policy=target_policy,
#     zf1=zf1, zf2=zf2, target_zf1=target_zf1, target_zf2=target_zf2,
#     discount=0.99, reward_scale=0.001,
#     policy_lr=3e-4, zf_lr=3e-4,
#     tau_type='iqn', num_quantiles=32,
#     risk_type='cvar', risk_param=0.25,
#     soft_target_tau=0.005,
#     use_automatic_entropy_tuning=False,
#     alpha=0.2
# )

# algorithm = TorchBatchRLAlgorithm(
#     trainer=trainer,
#     exploration_env=env, evaluation_env=eval_env,
#     exploration_data_collector=expl_collector,
#     evaluation_data_collector=eval_collector,
#     replay_buffer=replay_buffer,
#     batch_size=256, max_path_length=500,
#     num_epochs=100,
#     num_eval_steps_per_epoch=500,
#     num_expl_steps_per_train_loop=500,
#     num_trains_per_train_loop=500,
#     min_num_steps_before_training=5000,
# )

# # ── Step 5: train ────────────────────────────────────────────────
# gt.reset_root()
# setup_logger('dsac_cvar_finrl', variant={})
# algorithm.to(ptu.device)
# algorithm.train()

In [ ]:
# import rlkit.torch.dsac.dsac as dsac_module
# import torch

# original_train_from_torch = dsac_module.DSACTrainer.train_from_torch

# def patched_train_from_torch(self, batch):
#     # Convert all batch values to tensors before training
#     tensor_batch = {}
#     for k, v in batch.items():
#         if not isinstance(v, torch.Tensor):
#             tensor_batch[k] = torch.FloatTensor(v).to(ptu.device)
#         else:
#             tensor_batch[k] = v.to(ptu.device)
#     return original_train_from_torch(self, tensor_batch)

# dsac_module.DSACTrainer.train_from_torch = patched_train_from_torch

# # Remove the Linear patch now that we found the real source
# import torch.nn as nn
# nn.Linear.forward = original_linear_forward

In [ ]:
# import traceback
# import torch.nn as nn

# original_linear_forward = nn.Linear.forward

# def patched_linear_forward(self, input):
#     if not isinstance(input, torch.Tensor):
#         print(f"NON-TENSOR INPUT to Linear: type={type(input)}, value=\n{input}")
#         traceback.print_stack(limit=8)
#         input = torch.FloatTensor(input).to(next(self.parameters()).device)
#     return original_linear_forward(self, input)

# nn.Linear.forward = patched_linear_forward

In [ ]:
# def quantile_regression_loss(input, target, tau, weight):
#     """
#     input: (N, T)
#     target: (N, T)
#     tau: (N, T)
#     """
#     input = input.unsqueeze(-1)
#     target = target.detach().unsqueeze(-2)
#     tau = tau.detach().unsqueeze(-1)
#     weight = weight.detach().unsqueeze(-2)
#     expanded_input, expanded_target = torch.broadcast_tensors(input, target)
#     L = F.smooth_l1_loss(expanded_input, expanded_target, reduction="none")  # (N, T, T)
#     sign = torch.sign(expanded_input - expanded_target) / 2. + 0.5
#     rho = torch.abs(tau - sign) * L * weight
#     return rho.sum(dim=-1).mean()


# class DSACTrainer(TorchTrainer):

#     def __init__(
#             self,
#             env,
#             policy,
#             target_policy,
#             zf1,
#             zf2,
#             target_zf1,
#             target_zf2,
#             fp=None,
#             target_fp=None,
#             discount=0.99,
#             reward_scale=1.0,
#             alpha=1.0,
#             policy_lr=3e-4,
#             zf_lr=3e-4,
#             tau_type='iqn',
#             fp_lr=1e-5,
#             num_quantiles=32,
#             risk_type='neutral',
#             risk_param=0.,
#             risk_param_final=None,
#             risk_schedule_timesteps=1,
#             optimizer_class=optim.Adam,
#             soft_target_tau=5e-3,
#             target_update_period=1,
#             clip_norm=0.,
#             use_automatic_entropy_tuning=False,
#             target_entropy=None,
#     ):
#         super().__init__()
#         self.env = env
#         self.policy = policy
#         self.target_policy = target_policy
#         self.zf1 = zf1
#         self.zf2 = zf2
#         self.target_zf1 = target_zf1
#         self.target_zf2 = target_zf2

#         self.soft_target_tau = soft_target_tau
#         self.target_update_period = target_update_period
#         self.tau_type = tau_type
#         self.num_quantiles = num_quantiles

#         self.use_automatic_entropy_tuning = use_automatic_entropy_tuning
#         if self.use_automatic_entropy_tuning:
#             if target_entropy:
#                 self.target_entropy = target_entropy
#             else:
#                 self.target_entropy = -np.prod(self.env.action_space.shape).item()  # heuristic value from Tuomas
#             self.log_alpha = ptu.zeros(1, requires_grad=True)
#             self.alpha_optimizer = optimizer_class(
#                 [self.log_alpha],
#                 lr=policy_lr,
#             )
#         else:
#             self.alpha = alpha

#         self.zf_criterion = quantile_regression_loss

#         self.policy_optimizer = optimizer_class(
#             self.policy.parameters(),
#             lr=policy_lr,
#         )

#         self.zf1_optimizer = optimizer_class(
#             self.zf1.parameters(),
#             lr=zf_lr,
#         )
#         self.zf2_optimizer = optimizer_class(
#             self.zf2.parameters(),
#             lr=zf_lr,
#         )

#         self.fp = fp
#         self.target_fp = target_fp
#         if self.tau_type == 'fqf':
#             self.fp_optimizer = optimizer_class(
#                 self.fp.parameters(),
#                 lr=fp_lr,
#             )

#         self.discount = discount
#         self.reward_scale = reward_scale
#         self.clip_norm = clip_norm

#         self.risk_type = risk_type
#         self.risk_schedule = LinearSchedule(risk_schedule_timesteps, risk_param,
#                                             risk_param if risk_param_final is None else risk_param_final)

#         self.eval_statistics = OrderedDict()
#         self._n_train_steps_total = 0
#         self._need_to_update_eval_statistics = True

#     def get_tau(self, obs, actions, fp=None):
#         if self.tau_type == 'fix':
#             presum_tau = ptu.zeros(len(actions), self.num_quantiles) + 1. / self.num_quantiles
#         elif self.tau_type == 'iqn':  # add 0.1 to prevent tau getting too close
#             presum_tau = ptu.rand(len(actions), self.num_quantiles) + 0.1
#             presum_tau /= presum_tau.sum(dim=-1, keepdims=True)
#         elif self.tau_type == 'fqf':
#             if fp is None:
#                 fp = self.fp
#             presum_tau = fp(obs, actions)
#         tau = torch.cumsum(presum_tau, dim=1)  # (N, T), note that they are tau1...tauN in the paper
#         with torch.no_grad():
#             tau_hat = ptu.zeros_like(tau)
#             tau_hat[:, 0:1] = tau[:, 0:1] / 2.
#             tau_hat[:, 1:] = (tau[:, 1:] + tau[:, :-1]) / 2.
#         return tau, tau_hat, presum_tau

#     def train_from_torch(self, batch):
#         rewards = batch['rewards']
#         terminals = batch['terminals']
#         obs = batch['observations']
#         actions = batch['actions']
#         next_obs = batch['next_observations']
#         gt.stamp('preback_start', unique=False)
#         """
#         Update Alpha
#         """
#         new_actions, policy_mean, policy_log_std, log_pi, *_ = self.policy(
#             obs,
#             reparameterize=True,
#             return_log_prob=True,
#         )
#         if self.use_automatic_entropy_tuning:
#             alpha_loss = -(self.log_alpha.exp() * (log_pi + self.target_entropy).detach()).mean()
#             self.alpha_optimizer.zero_grad()
#             alpha_loss.backward()
#             self.alpha_optimizer.step()
#             alpha = self.log_alpha.exp()
#         else:
#             alpha_loss = 0
#             alpha = self.alpha
#         gt.stamp('preback_alpha', unique=False)
#         """
#         Update ZF
#         """
#         with torch.no_grad():
#             new_next_actions, _, _, new_log_pi, *_ = self.target_policy(
#                 next_obs,
#                 reparameterize=True,
#                 return_log_prob=True,
#             )
#             next_tau, next_tau_hat, next_presum_tau = self.get_tau(next_obs, new_next_actions, fp=self.target_fp)
#             target_z1_values = self.target_zf1(next_obs, new_next_actions, next_tau_hat)
#             target_z2_values = self.target_zf2(next_obs, new_next_actions, next_tau_hat)
#             target_z_values = torch.min(target_z1_values, target_z2_values) - alpha * new_log_pi
#             z_target = self.reward_scale * rewards + (1. - terminals) * self.discount * target_z_values

#         tau, tau_hat, presum_tau = self.get_tau(obs, actions, fp=self.fp)
#         z1_pred = self.zf1(obs, actions, tau_hat)
#         z2_pred = self.zf2(obs, actions, tau_hat)
#         zf1_loss = self.zf_criterion(z1_pred, z_target, tau_hat, next_presum_tau)
#         zf2_loss = self.zf_criterion(z2_pred, z_target, tau_hat, next_presum_tau)
#         gt.stamp('preback_zf', unique=False)

#         self.zf1_optimizer.zero_grad()
#         zf1_loss.backward()
#         self.zf1_optimizer.step()
#         gt.stamp('backward_zf1', unique=False)

#         self.zf2_optimizer.zero_grad()
#         zf2_loss.backward()
#         self.zf2_optimizer.step()
#         gt.stamp('backward_zf2', unique=False)
#         """
#         Update FP
#         """
#         if self.tau_type == 'fqf':
#             with torch.no_grad():
#                 dWdtau = 0.5 * (2 * self.zf1(obs, actions, tau[:, :-1]) - z1_pred[:, :-1] - z1_pred[:, 1:] +
#                                 2 * self.zf2(obs, actions, tau[:, :-1]) - z2_pred[:, :-1] - z2_pred[:, 1:])
#                 dWdtau /= dWdtau.shape[0]  # (N, T-1)
#             gt.stamp('preback_fp', unique=False)

#             self.fp_optimizer.zero_grad()
#             tau[:, :-1].backward(gradient=dWdtau)
#             self.fp_optimizer.step()
#             gt.stamp('backward_fp', unique=False)
#         """
#         Update Policy
#         """
#         risk_param = self.risk_schedule(self._n_train_steps_total)

#         if self.risk_type == 'VaR':
#             tau_ = ptu.ones_like(rewards) * risk_param
#             q1_new_actions = self.zf1(obs, new_actions, tau_)
#             q2_new_actions = self.zf2(obs, new_actions, tau_)
#         else:
#             with torch.no_grad():
#                 new_tau, new_tau_hat, new_presum_tau = self.get_tau(obs, new_actions, fp=self.fp)
#             z1_new_actions = self.zf1(obs, new_actions, new_tau_hat)
#             z2_new_actions = self.zf2(obs, new_actions, new_tau_hat)
#             if self.risk_type in ['neutral', 'std']:
#                 q1_new_actions = torch.sum(new_presum_tau * z1_new_actions, dim=1, keepdims=True)
#                 q2_new_actions = torch.sum(new_presum_tau * z2_new_actions, dim=1, keepdims=True)
#                 if self.risk_type == 'std':
#                     q1_std = new_presum_tau * (z1_new_actions - q1_new_actions).pow(2)
#                     q2_std = new_presum_tau * (z2_new_actions - q2_new_actions).pow(2)
#                     q1_new_actions -= risk_param * q1_std.sum(dim=1, keepdims=True).sqrt()
#                     q2_new_actions -= risk_param * q2_std.sum(dim=1, keepdims=True).sqrt()
#             else:
#                 with torch.no_grad():
#                     risk_weights = distortion_de(new_tau_hat, self.risk_type, risk_param)
#                 q1_new_actions = torch.sum(risk_weights * new_presum_tau * z1_new_actions, dim=1, keepdims=True)
#                 q2_new_actions = torch.sum(risk_weights * new_presum_tau * z2_new_actions, dim=1, keepdims=True)
#         q_new_actions = torch.min(q1_new_actions, q2_new_actions)

#         policy_loss = (alpha * log_pi - q_new_actions).mean()
#         gt.stamp('preback_policy', unique=False)

#         self.policy_optimizer.zero_grad()
#         policy_loss.backward()
#         policy_grad = ptu.fast_clip_grad_norm(self.policy.parameters(), self.clip_norm)
#         self.policy_optimizer.step()
#         gt.stamp('backward_policy', unique=False)
#         """
#         Soft Updates
#         """
#         if self._n_train_steps_total % self.target_update_period == 0:
#             ptu.soft_update_from_to(self.policy, self.target_policy, self.soft_target_tau)
#             ptu.soft_update_from_to(self.zf1, self.target_zf1, self.soft_target_tau)
#             ptu.soft_update_from_to(self.zf2, self.target_zf2, self.soft_target_tau)
#             if self.tau_type == 'fqf':
#                 ptu.soft_update_from_to(self.fp, self.target_fp, self.soft_target_tau)
#         """
#         Save some statistics for eval
#         """
#         if self._need_to_update_eval_statistics:
#             self._need_to_update_eval_statistics = False
#             """
#             Eval should set this to None.
#             This way, these statistics are only computed for one batch.
#             """
#             policy_loss = (log_pi - q_new_actions).mean()

#             self.eval_statistics['ZF1 Loss'] = zf1_loss.item()
#             self.eval_statistics['ZF2 Loss'] = zf2_loss.item()
#             self.eval_statistics['Policy Loss'] = policy_loss.item()
#             self.eval_statistics['Policy Grad'] = policy_grad
#             self.eval_statistics.update(create_stats_ordered_dict(
#                 'Z1 Predictions',
#                 ptu.get_numpy(z1_pred),
#             ))
#             self.eval_statistics.update(create_stats_ordered_dict(
#                 'Z2 Predictions',
#                 ptu.get_numpy(z2_pred),
#             ))
#             self.eval_statistics.update(create_stats_ordered_dict(
#                 'Z Targets',
#                 ptu.get_numpy(z_target),
#             ))
#             self.eval_statistics.update(create_stats_ordered_dict(
#                 'Log Pis',
#                 ptu.get_numpy(log_pi),
#             ))
#             self.eval_statistics.update(create_stats_ordered_dict(
#                 'Policy mu',
#                 ptu.get_numpy(policy_mean),
#             ))
#             self.eval_statistics.update(create_stats_ordered_dict(
#                 'Policy log std',
#                 ptu.get_numpy(policy_log_std),
#             ))

#             if self.use_automatic_entropy_tuning:
#                 self.eval_statistics['Alpha'] = alpha.item()
#                 self.eval_statistics['Alpha Loss'] = alpha_loss.item()
#         self._n_train_steps_total += 1

#     def get_diagnostics(self):
#         return self.eval_statistics

#     def end_epoch(self, epoch):
#         self._need_to_update_eval_statistics = True

#     @property
#     def networks(self):
#         networks = [
#             self.policy,
#             self.target_policy,
#             self.zf1,
#             self.zf2,
#             self.target_zf1,
#             self.target_zf2,
#         ]
#         if self.tau_type == 'fqf':
#             networks += [
#                 self.fp,
#                 self.target_fp,
#             ]
#         return networks

#     def get_snapshot(self):
#         snapshot = dict(
#             policy=self.policy.state_dict(),
#             target_policy=self.target_policy.state_dict(),
#             zf1=self.zf1.state_dict(),
#             zf2=self.zf2.state_dict(),
#             target_zf1=self.target_zf1.state_dict(),
#             target_zf2=self.target_zf2.state_dict(),
#         )
#         if self.tau_type == 'fqf':
#             snapshot['fp'] = self.fp.state_dict()
#             snapshot['target_fp'] = self.target_fp.state_dict()
#         return snapshot

In [ ]:
stock_dim   = len(TICKER_LIST)
state_space = 1 + 2 * stock_dim + len(INDICATORS) * stock_dim

ENV_KWARGS = dict(
    hmax=100,
    initial_amount=1_000_000,
    num_stock_shares=[0] * stock_dim,
    buy_cost_pct=[0.001] * stock_dim,
    sell_cost_pct=[0.001] * stock_dim,
    reward_scaling=1e-4,
    state_space=state_space,
    stock_dim=stock_dim,
    tech_indicator_list=INDICATORS,
    action_space=stock_dim,
    make_plots=False,
    print_verbosity=10,
)

train_env = StockTradingEnv(df=df_train, **ENV_KWARGS)
test_env  = StockTradingEnv(df=df_test,  **ENV_KWARGS)

state_dim  = train_env.observation_space.shape[0]
action_dim = train_env.action_space.shape[0]
action_scale = float(train_env.action_space.high[0])

print(f'State dim: {state_dim} | Action dim: {action_dim} | Action scale: {action_scale}')

In [ ]:
# # ──────────────────────────────────────────────
# # 5.1  Replay Buffer
# # ──────────────────────────────────────────────
# class ReplayBuffer:
#     def __init__(self, capacity=200_000):
#         self.buffer = deque(maxlen=capacity)

#     def push(self, state, action, reward, next_state, done):
#         self.buffer.append((state, action, reward, next_state, done))

#     def sample(self, batch_size):
#         batch = random.sample(self.buffer, batch_size)
#         s, a, r, ns, d = zip(*batch)
#         return (
#             torch.FloatTensor(np.array(s)).to(device),
#             torch.FloatTensor(np.array(a)).to(device),
#             torch.FloatTensor(np.array(r)).unsqueeze(1).to(device),
#             torch.FloatTensor(np.array(ns)).to(device),
#             torch.FloatTensor(np.array(d)).unsqueeze(1).to(device),
#         )

#     def __len__(self):
#         return len(self.buffer)


# # ──────────────────────────────────────────────
# # 5.2  IQN Critic  (distributional)
# # ──────────────────────────────────────────────
# class IQNCritic(nn.Module):
#     """
#     Implicit Quantile Network critic.
#     Maps (state, action, τ) → Z(s,a) at quantile τ.
#     """
#     def __init__(self, state_dim, action_dim, hidden=256, n_cos=64):
#         super().__init__()
#         self.n_cos = n_cos
#         self.hidden = hidden

#         # Feature extractor
#         self.feature_net = nn.Sequential(
#             nn.Linear(state_dim + action_dim, hidden),
#             nn.ReLU(),
#         )
#         # Cosine embedding for quantile τ
#         self.cos_net = nn.Sequential(
#             nn.Linear(n_cos, hidden),
#             nn.ReLU(),
#         )
#         # Output head
#         self.out_net = nn.Sequential(
#             nn.Linear(hidden, hidden),
#             nn.ReLU(),
#             nn.Linear(hidden, 1),
#         )
#         # Register cosine basis
#         i = torch.arange(1, n_cos + 1, dtype=torch.float32).unsqueeze(0)  # (1, n_cos)
#         self.register_buffer('i_pi', i * np.pi)

#     def forward(self, state, action, taus):
#         """
#         state:  (B, state_dim)
#         action: (B, action_dim)
#         taus:   (B, N)  — quantile levels in [0,1]
#         returns quantiles: (B, N)
#         """
#         B, N = taus.shape
#         sa = torch.cat([state, action], dim=-1)          # (B, s+a)
#         h  = self.feature_net(sa)                        # (B, hidden)

#         # Cosine embedding
#         taus_flat = taus.reshape(B * N, 1)               # (B*N, 1)
#         cos_emb   = torch.cos(taus_flat * self.i_pi)     # (B*N, n_cos)
#         tau_emb   = self.cos_net(cos_emb)                # (B*N, hidden)

#         # Hadamard product
#         h_exp = h.unsqueeze(1).expand(B, N, self.hidden).reshape(B * N, self.hidden)
#         merged = h_exp * tau_emb                         # (B*N, hidden)
#         q = self.out_net(merged).reshape(B, N)           # (B, N)
#         return q


# # ──────────────────────────────────────────────
# # 5.3  Gaussian Actor
# # ──────────────────────────────────────────────
# class GaussianActor(nn.Module):
#     LOG_STD_MIN, LOG_STD_MAX = -20, 2

#     def __init__(self, state_dim, action_dim, hidden=256):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(state_dim, hidden), nn.ReLU(),
#             nn.Linear(hidden, hidden),   nn.ReLU(),
#         )
#         self.mean_head    = nn.Linear(hidden, action_dim)
#         self.log_std_head = nn.Linear(hidden, action_dim)

#     def forward(self, state):
#         h       = self.net(state)
#         mean    = self.mean_head(h)
#         log_std = self.log_std_head(h).clamp(self.LOG_STD_MIN, self.LOG_STD_MAX)
#         return mean, log_std

#     def sample(self, state):
#         mean, log_std = self.forward(state)
#         std  = log_std.exp()
#         dist = Normal(mean, std)
#         x_t  = dist.rsample()
#         y_t  = torch.tanh(x_t)
#         log_prob = dist.log_prob(x_t) - torch.log(1 - y_t.pow(2) + 1e-6)
#         log_prob = log_prob.sum(dim=-1, keepdim=True)
#         return y_t, log_prob, torch.tanh(mean)

In [ ]:
# def compute_cvar(quantiles: torch.Tensor, alpha: float) -> torch.Tensor:
#     """
#     CVaR at confidence level alpha (risk-averse when alpha < 1).

#     CVaR_α = E[Z | Z ≤ VaR_α(Z)]
#            = (1/α) * ∫₀^α z(τ) dτ

#     Since we sample N quantiles uniformly in [0,1], we approximate:
#         CVaR_α ≈ mean of the lowest α*N quantile values

#     Args:
#         quantiles: (B, N) — sampled return quantiles
#         alpha:     float in (0, 1] — lower tail fraction
#                    alpha=1.0 → risk-neutral (mean)
#                    alpha=0.1 → worst 10% tail
#     Returns:
#         cvar: (B, 1)
#     """
#     batch_size, N = quantiles.shape
#     num_tails = int(alpha * N)
#     if num_tails < 1:
#         num_tails = 1

#     # Sort and take the lowest 'num_tails' values
#     sorted_q, _ = torch.sort(quantiles, dim=1)
#     cvar = sorted_q[:, :num_tails].mean(dim=1, keepdim=True)
#     return cvar

In [ ]:
# class DSACAgent:
#     def __init__(
#         self,
#         state_dim,
#         action_dim,
#         action_scale,
#         # CVaR
#         cvar_alpha       = 0.25,   # optimize worst 25% tail
#         # IQN
#         n_quantiles      = 32,     # N quantiles sampled per update
#         n_target_quantiles = 32,
#         # SAC
#         gamma            = 0.99,
#         tau              = 0.005,
#         alpha_lr         = 3e-4,
#         actor_lr         = 3e-4,
#         critic_lr        = 3e-4,
#         batch_size       = 256,
#         hidden           = 256,
#         auto_entropy     = True,
#         target_entropy   = None,
#     ):
#         self.action_scale = action_scale
#         self.gamma        = gamma
#         self.tau          = tau
#         self.batch_size   = batch_size
#         self.cvar_alpha   = cvar_alpha
#         self.N            = n_quantiles
#         self.N_target     = n_target_quantiles
#         self.auto_entropy = auto_entropy

#         # ── Networks ──────────────────────────────────
#         self.actor    = GaussianActor(state_dim, action_dim, hidden).to(device)
#         self.critic1  = IQNCritic(state_dim, action_dim, hidden).to(device)
#         self.critic2  = IQNCritic(state_dim, action_dim, hidden).to(device)
#         self.target1  = IQNCritic(state_dim, action_dim, hidden).to(device)
#         self.target2  = IQNCritic(state_dim, action_dim, hidden).to(device)
#         self.target1.load_state_dict(self.critic1.state_dict())
#         self.target2.load_state_dict(self.critic2.state_dict())

#         # ── Entropy temperature ────────────────────────
#         if target_entropy is None:
#             self.target_entropy = -action_dim
#         else:
#             self.target_entropy = target_entropy
#         self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
#         self.alpha_val = self.log_alpha.exp().item()

#         # ── Optimizers ────────────────────────────────
#         self.actor_opt   = optim.Adam(self.actor.parameters(),   lr=actor_lr)
#         self.critic1_opt = optim.Adam(self.critic1.parameters(), lr=critic_lr)
#         self.critic2_opt = optim.Adam(self.critic2.parameters(), lr=critic_lr)
#         self.alpha_opt   = optim.Adam([self.log_alpha],          lr=alpha_lr)

#         self.replay = ReplayBuffer()

#     # ── Helpers ────────────────────────────────────────
#     def _sample_taus(self, B, N):
#         return torch.rand(B, N, device=device)

#     def _soft_update(self, net, target):
#         for p, tp in zip(net.parameters(), target.parameters()):
#             tp.data.copy_(self.tau * p.data + (1 - self.tau) * tp.data)

#     # ── Action selection ──────────────────────────────
#     def select_action(self, state, deterministic=False):
#         with torch.no_grad():
#             state = torch.FloatTensor(state).unsqueeze(0).to(device)
#             if deterministic:
#                 _, _, mean = self.actor.sample(state)
#                 return (mean * self.action_scale).cpu().numpy()[0]
#             action, _, _ = self.actor.sample(state)
#             return (action * self.action_scale).cpu().numpy()[0]

#     # ── Quantile Huber Loss ───────────────────────────
#     def _quantile_huber_loss(self, quantiles, target_quantiles, taus, kappa=1.0):
#         B, N  = quantiles.shape
#         N_    = target_quantiles.shape[1]

#         q  = quantiles.unsqueeze(2)                            # (B, N, 1)
#         tq = target_quantiles.detach().unsqueeze(1)            # (B, 1, N')
#         u  = tq - q                                            # (B, N, N')

#         huber = torch.where(u.abs() <= kappa,
#                             0.5 * u.pow(2),
#                             kappa * (u.abs() - 0.5 * kappa))

#         tau_exp = taus.detach().unsqueeze(2).expand(B, N, N_)  # (B, N, N')
#         loss = (torch.abs(tau_exp - (u.detach() < 0).float()) * huber).mean(dim=2).mean(dim=1)
#         return loss.mean()

#     # ── Update step ───────────────────────────────────
#     def update(self):
#         if len(self.replay) < self.batch_size:
#             return {}
#         with torch.enable_grad():
#             states, actions, rewards, next_states, dones = self.replay.sample(self.batch_size)
#             B = states.shape[0]

#             # ── Target quantiles ──────────────────────────
#             with torch.no_grad():
#                 next_actions, next_log_pi, _ = self.actor.sample(next_states)
#                 taus_next = self._sample_taus(B, self.N_target)
#                 q1_next = self.target1(next_states, next_actions, taus_next)
#                 q2_next = self.target2(next_states, next_actions, taus_next)
#                 q_next  = torch.min(q1_next, q2_next)
#                 alpha_val = self.log_alpha.exp().item()
#                 q_next    = q_next - alpha_val * next_log_pi
#                 target_q  = rewards + (1 - dones) * self.gamma * q_next

#             # ── Critic update ─────────────────────────────
#             taus1 = self._sample_taus(B, self.N)
#             taus2 = self._sample_taus(B, self.N)
#             q1 = self.critic1(states, actions, taus1)
#             q2 = self.critic2(states, actions, taus2)

#             critic1_loss = self._quantile_huber_loss(q1, target_q, taus1)
#             critic2_loss = self._quantile_huber_loss(q2, target_q, taus2)

#             self.critic1_opt.zero_grad(); critic1_loss.backward(); self.critic1_opt.step()
#             self.critic2_opt.zero_grad(); critic2_loss.backward(); self.critic2_opt.step()

#             # ── Actor update (CVaR objective) ─────────────
#             pi, log_pi, _ = self.actor.sample(states)

#             # We need gradients to flow through 'pi' into the actor's parameters.
#             # We do NOT want gradients to flow into the critic's parameters here.
#             taus_actor = self._sample_taus(B, self.N)
#             q1_pi = self.critic1(states, pi, taus_actor)
#             q2_pi = self.critic2(states, pi, taus_actor)
#             q_pi  = torch.min(q1_pi, q2_pi)

#             # Compute CVaR (ensure compute_cvar is differentiable)
#             cvar = compute_cvar(q_pi, self.cvar_alpha)

#             # The loss now has a valid grad_fn connecting it to the actor
#             actor_loss = (alpha_val * log_pi - cvar).mean()

#             self.actor_opt.zero_grad()
#             actor_loss.backward()
#             self.actor_opt.step()
#             # ── Entropy temperature update ─────────────────
#             if self.auto_entropy:
#                 _, log_pi_new, _ = self.actor.sample(states)
#                 alpha_loss = -(self.log_alpha.exp() * (log_pi_new + self.target_entropy).detach()).mean()
#                 self.alpha_opt.zero_grad(); alpha_loss.backward(); self.alpha_opt.step()
#                 self.alpha_val = self.log_alpha.exp().item()

#             # ── Soft update targets ────────────────────────
#             self._soft_update(self.critic1, self.target1)
#             self._soft_update(self.critic2, self.target2)

#             return {
#                 'critic1_loss': critic1_loss.item(),
#                 'critic2_loss': critic2_loss.item(),
#                 'actor_loss':   actor_loss.item(),
#                 'alpha':        self.alpha_val,
#             }

In [ ]:
# # ── Hyperparameters ──────────────────────────────────────────────────
# CVAR_ALPHA        = 0.25   # Optimise worst 25% tail (lower = more risk-averse)
# N_EPISODES        = 50
# MAX_STEPS         = 500    # max steps per episode (None = full episode)
# WARMUP_STEPS      = 5000   # random actions before training
# UPDATE_FREQ       = 1      # update every step
# GRADIENT_STEPS    = 1
# EVAL_EVERY        = 5      # evaluate on test env every N episodes

# # ── Create agent ─────────────────────────────────────────────────────
# agent = DSACAgent(
#     state_dim    = state_dim,
#     action_dim   = action_dim,
#     action_scale = action_scale,
#     cvar_alpha   = CVAR_ALPHA,
#     n_quantiles  = 32,
#     gamma        = 0.99,
#     tau          = 0.005,
#     batch_size   = 256,
#     hidden       = 256,
# )

# train_rewards = []
# eval_rewards  = []
# total_steps   = 0

# print(f'Training DSAC with CVaR α={CVAR_ALPHA} for {N_EPISODES} episodes...')
# print('=' * 60)

# for episode in range(1, N_EPISODES + 1):
#     state, _ = train_env.reset()
#     ep_reward = 0
#     step = 0

#     while True:
#         # Action selection
#         if total_steps < WARMUP_STEPS:
#             action = train_env.action_space.sample()
#         else:
#             action = agent.select_action(state)

#         next_state, reward, terminated, truncated, _ = train_env.step(action)
#         done = terminated or truncated

#         agent.replay.push(state, action, reward, next_state, float(done))

#         if total_steps >= WARMUP_STEPS and total_steps % UPDATE_FREQ == 0:
#             for _ in range(GRADIENT_STEPS):
#                 agent.update()

#         state       = next_state
#         ep_reward  += reward
#         total_steps += 1
#         step        += 1

#         if done or (MAX_STEPS and step >= MAX_STEPS):
#             break

#     train_rewards.append(ep_reward)

#     # ── Evaluation ────────────────────────────────────────────────
#     if episode % EVAL_EVERY == 0:
#         eval_state, _ = test_env.reset()
#         eval_reward   = 0
#         while True:
#             eval_action = agent.select_action(eval_state, deterministic=True)
#             eval_state, r, term, trunc, _ = test_env.step(eval_action)
#             eval_reward += r
#             if term or trunc:
#                 break
#         eval_rewards.append(eval_reward)
#         print(f'Ep {episode:3d} | Train R: {ep_reward:8.2f} | '
#               f'Eval R: {eval_reward:8.2f} | Steps: {total_steps}')
#     else:
#         print(f'Ep {episode:3d} | Train R: {ep_reward:8.2f} | Steps: {total_steps}')

# print('\nTraining complete!')

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# axes[0].plot(train_rewards, label='Train', color='steelblue', alpha=0.8)
# # smoothed
# window = min(5, len(train_rewards))
# smooth = pd.Series(train_rewards).rolling(window).mean()
# axes[0].plot(smooth, color='navy', linewidth=2, label=f'Smoothed ({window}-ep)')
# axes[0].set_title(f'DSAC + CVaR (α={CVAR_ALPHA}) — Training Rewards')
# axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Episode Reward')
# axes[0].legend(); axes[0].grid(True, alpha=0.3)

# if eval_rewards:
#     eval_eps = list(range(EVAL_EVERY, N_EPISODES + 1, EVAL_EVERY))
#     axes[1].plot(eval_eps, eval_rewards, 'o-', color='darkorange', label='Eval')
#     axes[1].set_title('Evaluation Rewards (Test Environment)')
#     axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Episode Reward')
#     axes[1].legend(); axes[1].grid(True, alpha=0.3)

# plt.tight_layout()
# plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
# plt.show()

In [ ]:
# state, _ = test_env.reset()
# portfolio_values = [test_env.initial_amount]
# done = False

# while not done:
#     action = agent.select_action(state, deterministic=True)
#     state, reward, terminated, truncated, info = test_env.step(action)
#     done = terminated or truncated
#     portfolio_values.append(test_env.asset_memory[-1])

# portfolio_values = np.array(portfolio_values)
# returns = np.diff(portfolio_values) / portfolio_values[:-1]

# total_return   = (portfolio_values[-1] / portfolio_values[0] - 1) * 100
# ann_return     = ((portfolio_values[-1] / portfolio_values[0]) ** (252 / len(returns)) - 1) * 100
# sharpe         = np.mean(returns) / (np.std(returns) + 1e-9) * np.sqrt(252)
# max_dd         = np.min(portfolio_values / np.maximum.accumulate(portfolio_values) - 1) * 100
# alpha_var      = CVAR_ALPHA
# var_95         = np.percentile(returns, 5) * 100
# cvar_95        = returns[returns <= np.percentile(returns, 5)].mean() * 100

# print('=' * 45)
# print('         BACKTEST RESULTS (Test Set)')
# print('=' * 45)
# print(f'  Total Return    : {total_return:+.2f}%')
# print(f'  Annualised Ret  : {ann_return:+.2f}%')
# print(f'  Sharpe Ratio    : {sharpe:.3f}')
# print(f'  Max Drawdown    : {max_dd:.2f}%')
# print(f'  VaR (5%)        : {var_95:.4f}%')
# print(f'  CVaR (5%)       : {cvar_95:.4f}%  ← tail risk')
# print(f'  CVaR α used     : {alpha_var}')
# print('=' * 45)

# # Plot portfolio curve
# plt.figure(figsize=(12, 4))
# plt.plot(portfolio_values / portfolio_values[0], color='teal', linewidth=1.5)
# plt.axhline(1.0, color='grey', linestyle='--', linewidth=0.8)
# plt.title(f'DSAC + CVaR (α={CVAR_ALPHA}) — Portfolio Value (Test)')
# plt.xlabel('Timestep'); plt.ylabel('Normalised Portfolio Value')
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.savefig('portfolio_curve.png', dpi=150, bbox_inches='tight')
# plt.show()